# 04 — Customer Segmentation

**Purpose:** Group customers into meaningful behavioural segments using K-Means clustering.

---

## What is K-Means Clustering?

K-Means is an **unsupervised** machine learning algorithm. Unlike classification (where we have labels), clustering finds natural groups in the data on its own.

**How it works:**
1. Pick K random starting points (centroids)
2. Assign every customer to the nearest centroid
3. Move each centroid to the average of its assigned customers
4. Repeat steps 2-3 until nothing changes

## Why Standardise Features?

K-Means uses **Euclidean distance**. If `monthly_fee` ranges from 5–80 and `support_tickets` ranges from 0–10, the fee dominates the distance calculation just because it's bigger numerically. `StandardScaler` converts all features to mean=0, std=1 so they're on equal footing.

## What is RFM?

**Recency** — How recently was the customer active? (lower = better)  
**Frequency** — How often do they use the product? (higher = better)  
**Monetary** — How much do they pay? (higher = more valuable)  

RFM is a classic marketing framework for understanding customer value.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from data_processing import load_clean_data
from feature_engineering import build_rfm_features
from segmentation import (
    build_segmentation_features, find_optimal_k,
    fit_kmeans, profile_clusters, assign_segment_labels
)

df = load_clean_data()
print(f'Loaded {len(df):,} customers')

## 1. Build Segmentation Features

In [ ]:
features = build_segmentation_features(df)
print('Features for clustering:')
print(features.describe().round(2))

## 2. Find Optimal K — Elbow Method + Silhouette Score

**Elbow Method:** Plot inertia (sum of squared distances to centroids) for each K. The "elbow" — where the drop flattens — suggests the optimal K.

**Silhouette Score:** Measures cluster quality. Range: -1 to +1. Higher is better.

In [ ]:
k_results = find_optimal_k(features, k_range=range(2, 9))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(k_results['k_values'], k_results['inertias'], 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method — Finding Optimal K', fontsize=13)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (lower = tighter clusters)')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Chosen K=4')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(k_results['k_values'], k_results['silhouette_scores'], 'go-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score — Cluster Quality', fontsize=13)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Chosen K=4')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

for k, s in zip(k_results['k_values'], k_results['silhouette_scores']):
    print(f'  K={k}: silhouette={s:.3f}')

## 3. Fit K-Means with K=4

In [ ]:
labels, km, scaler = fit_kmeans(features, n_clusters=4)

df_seg = df.loc[features.index].copy()
df_seg['cluster'] = labels

print('Cluster sizes:')
print(pd.Series(labels).value_counts().sort_index())

## 4. Profile the Clusters

**Important:** We look at the data FIRST, then assign names. Labels come from analysis, not assumptions.

In [ ]:
feature_cols = list(features.columns)
profile = profile_clusters(df_seg, feature_cols)
print('Cluster profiles (mean values):')
profile

In [ ]:
# Assign business-friendly labels based on profile
segment_map = assign_segment_labels(profile)
df_seg['segment_label'] = df_seg['cluster'].map(segment_map)

print('Assigned segment labels:')
for k, v in segment_map.items():
    size = (df_seg['cluster'] == k).sum()
    churn = df_seg[df_seg['cluster'] == k]['churned'].mean() * 100
    print(f'  Cluster {k} → "{v}" | {size:,} customers | {churn:.1f}% churn rate')

## 5. Visualise Segments

In [ ]:
# Distribution
seg_counts = df_seg['segment_label'].value_counts().reset_index()
seg_counts.columns = ['Segment', 'Count']
fig = px.pie(seg_counts, names='Segment', values='Count',
             title='Customer Segment Distribution', hole=0.35)
fig.show()

In [ ]:
# Scatter: Usage vs Payment Failures, coloured by segment
fig = px.scatter(df_seg, x='usage_frequency', y='payment_failures',
                 color='segment_label', opacity=0.4,
                 title='Customer Segments: Usage vs Payment Failures',
                 labels={'usage_frequency': 'Usage Frequency', 'payment_failures': 'Payment Failures'})
fig.show()

In [ ]:
# Churn rate by segment
seg_churn = df_seg.groupby('segment_label')['churned'].mean().reset_index()
seg_churn.columns = ['Segment', 'Churn Rate']
seg_churn['Churn Rate'] = (seg_churn['Churn Rate'] * 100).round(1)
seg_churn = seg_churn.sort_values('Churn Rate')

fig = px.bar(seg_churn, x='Churn Rate', y='Segment', orientation='h',
             title='Churn Rate by Customer Segment',
             color='Churn Rate', color_continuous_scale='RdYlGn_r', text='Churn Rate')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Business Takeaways

Each segment should receive different treatment:

| Segment | Strategy |
|---------|----------|
| **Loyal / High Value** | Reward programs, upsell, referral campaigns |
| **Average / Stable** | Feature education, gentle nudges to increase usage |
| **Low Engagement** | Re-engagement campaigns, usage tips, check-in calls |
| **High Risk / At Risk** | Immediate outreach, offers, payment issue resolution |